# Entrenamiento de Modelos de Machine Learning

1. **Preprocesar los datos de forma científica** usando Pipelines y ColumnTransformer de Scikit-Learn (para evitar la fuga de datos del conjunto de prueba).
2. **Entrenar y ajustar 5 algoritmos** (Regresión Logística, Árbol de Decisión, Random Forest, XGBoost y una Red Neuronal con Keras/TensorFlow).
3. **Serializar y guardar** los modelos para producción.


In [1]:
import sys
from pathlib import Path

import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

# Añadir directorio raíz al path para usar src
sys.path.append(str(Path.cwd().parent))

## 1. Cargar y Preprocesar los Datos

Primero dividimos los datos en **Entrenamiento (Train)** y **Prueba (Test)**.

El conjunto de prueba representa el futuro del modelo. Nunca debemos preprocesar todo el dataset a la vez: si calculamos la mediana de una columna sobre el dataset completo para rellenar nulos, el valor del test influiría en el de train, causando una fuga de información sutil.

> **Solución**: El preprocesador (ColumnTransformer) vive dentro del Pipeline. Al llamar a pipeline.fit(X_train, y_train), el Pipeline ajusta el preprocesador usando únicamente los datos de entrenamiento y lo aplica automáticamente al conjunto de prueba en la evaluación, sin contaminarlo.


In [2]:
from src.config import COLS_TO_DROP, MODELS_DIR, NA_TOKENS, RAW_DATA_PATH, TARGET_COL

# 1. Cargar dataset normalizando cadenas "NULL", "NA" y "" a NaN real
df = pd.read_csv(RAW_DATA_PATH, na_values=NA_TOKENS, keep_default_na=True)

# 2. Eliminar columnas irrelevantes o que causan fuga de datos
df_clean = df.drop(columns=[col for col in COLS_TO_DROP if col in df.columns])

# 3. Convertir agent a texto. Es un identificador de agencia sin orden numerico.
#    Los nulos se marcan como "Unknown" para que el imputer no reciba pd.NA.
if "agent" in df_clean.columns:
    agent_s = df_clean["agent"].astype("Int64").astype("object")
    df_clean["agent"] = agent_s.where(agent_s.notna(), other="Unknown").astype(str)

# 4. Separar X e y
X = df_clean.drop(columns=[TARGET_COL])
y = df_clean[TARGET_COL]

# 5. Dividir de forma estratificada (mismo ratio de cancelaciones en ambos conjuntos)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Conjunto de Entrenamiento: {X_train.shape[0]:,} filas")
print(f"Conjunto de Prueba: {X_test.shape[0]:,} filas")

Conjunto de Entrenamiento: 95,512 filas
Conjunto de Prueba: 23,878 filas


### Construcción del Preprocesador

Usamos las listas de columnas explícitas definidas en src/config.py en lugar de inferir los tipos por columna. Esto garantiza que el preprocesador sea reproducible y coherente con el pipeline de producción: las mismas columnas, en el mismo orden, siempre.

* Para numéricas: rellenar vacíos con la mediana y normalizar la escala (StandardScaler).
* Para categóricas: rellenar vacíos con "Unknown" y aplicar codificación binaria (OneHotEncoder).
* arrival_date_year se descarta mediante remainder="drop". El dataset cubre años parciales (2015 solo jul-dic, 2017 solo ene-ago), lo que genera una correlacion espuria con la variable objetivo que no se repetiría en producción.


In [3]:
from src.config import CAT_COLS, NUM_COLS

# Usar las listas explícitas de config en lugar de inferir tipos por columna.
# Esto garantiza que el preprocesador sea determinista y consistente con el
# pipeline de produccion: las mismas columnas, en el mismo orden, siempre.
num_cols = [c for c in NUM_COLS if c in X.columns]
cat_cols = [c for c in CAT_COLS if c in X.columns]

# Pipeline para variables numéricas
num_pipeline = Pipeline(
    [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

# Pipeline para variables categóricas
cat_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

# Combinar en un único ColumnTransformer.
# remainder="drop" descarta cualquier columna no declarada explícitamente
# (incluida arrival_date_year, que excluimos para evitar sesgo temporal).
preprocessor = ColumnTransformer(
    transformers=[("num", num_pipeline, num_cols), ("cat", cat_pipeline, cat_cols)],
    remainder="drop",
)

## 2. Entrenamiento de los 5 Modelos

Entrenaremos los 5 modelos encapsulando cada uno en un Pipeline que une el preprocesador con el clasificador. La red neuronal se incluye en el mismo bucle que el resto gracias a un adaptador que implementa la interfaz de Scikit-Learn (KerasMLPClassifier, definido en src/model_trainer.py).


In [4]:
from src.model_trainer import KerasMLPClassifier

# Diccionario de los 5 modelos con sus hiperparámetros.
# min_samples_leaf en los árboles evita ramas con muy pocas observaciones.
# subsample y colsample_bytree en XGBoost añaden regularización estocástica.
# scale_pos_weight=1.7 compensa el desbalanceo (~63% no cancel / ~37% cancel).
# KerasMLPClassifier adapta la red neuronal a la interfaz de Scikit-Learn
# e internamente aplica class_weight="balanced" para el mismo fin.
models = {
    "logistic_regression": LogisticRegression(max_iter=1000, random_state=42),
    "decision_tree": DecisionTreeClassifier(max_depth=12, min_samples_leaf=50, random_state=42),
    "random_forest": RandomForestClassifier(
        n_estimators=200, max_depth=15, min_samples_leaf=20, n_jobs=-1, random_state=42
    ),
    "xgboost": XGBClassifier(
        n_estimators=300,
        max_depth=10,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=1.7,
        n_jobs=-1,
        random_state=42,
    ),
    "neural_network": KerasMLPClassifier(epochs=30, batch_size=256, random_state=42),
}

trained_pipelines = {}

for name, model in models.items():
    print(f"Entrenando pipeline para {name.replace('_', ' ').title()}...")

    pipeline = Pipeline([("preprocessor", preprocessor), ("classifier", model)])
    pipeline.fit(X_train, y_train)
    trained_pipelines[name] = pipeline

    joblib.dump(pipeline, MODELS_DIR / f"{name}.pkl")
    print(f"Guardado en models/{name}.pkl\n")

Entrenando pipeline para Logistic Regression...
Guardado en models/logistic_regression.pkl

Entrenando pipeline para Decision Tree...
Guardado en models/decision_tree.pkl

Entrenando pipeline para Random Forest...
Guardado en models/random_forest.pkl

Entrenando pipeline para Xgboost...
Guardado en models/xgboost.pkl

Entrenando pipeline para Neural Network...
Guardado en models/neural_network.pkl



### Nota sobre la serialización

Todos los modelos, incluida la red neuronal, se guardan como un único archivo .pkl que contiene tanto el preprocesador como el clasificador. Esto simplifica la carga en produccion: un solo joblib.load es suficiente para obtener un objeto listo para recibir datos en crudo y devolver una prediccion.


## 3. Bonus — Ajuste de Hiperparámetros de XGBoost con RandomizedSearchCV

Los hiperparámetros fijados en el paso anterior son razonables, pero no necesariamente óptimos. **RandomizedSearchCV** explora aleatoriamente un espacio de combinaciones y selecciona la que maximiza el ROC-AUC en validación cruzada (3 folds), evitando el coste de un GridSearchCV exhaustivo.

> El preprocesador vive **dentro** del Pipeline que se pasa al search: esto garantiza que la normalización se ajusta solo con los datos de entrenamiento de cada fold, sin contaminar la validación.


In [5]:
import json
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "classifier__n_estimators": [200, 300, 400, 500],
    "classifier__max_depth": [6, 8, 10, 12],
    "classifier__learning_rate": [0.05, 0.08, 0.1, 0.15],
    "classifier__subsample": [0.7, 0.8, 0.9],
    "classifier__colsample_bytree": [0.7, 0.8, 0.9],
    "classifier__min_child_weight": [1, 3, 5],
    "classifier__gamma": [0, 0.1, 0.3],
    "classifier__reg_alpha": [0, 0.1, 0.5],
    "classifier__reg_lambda": [1, 1.5, 2],
}

# Base pipeline: preprocesador + XGBoost con scale_pos_weight
base_xgb_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(scale_pos_weight=1.7, n_jobs=-1, random_state=42)),
    ]
)

search = RandomizedSearchCV(
    base_xgb_pipeline,
    param_dist,
    n_iter=20,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=42,
    verbose=1,
    refit=True,
)
search.fit(X_train, y_train)

best_params = {k.replace("classifier__", ""): v for k, v in search.best_params_.items()}
print(f"\nMejores hiperparámetros: {best_params}")
print(f"Mejor ROC-AUC CV: {search.best_score_:.4f}")

# Guardar los mejores parámetros y el modelo tuneado
with open(MODELS_DIR.parent / "outputs" / "best_hiperparametros.json", "w") as f:
    json.dump(best_params, f, indent=2)

joblib.dump(search.best_estimator_, MODELS_DIR / "xgboost.pkl")
print("Modelo XGBoost tuneado guardado en models/xgboost.pkl")

Fitting 3 folds for each of 20 candidates, totalling 60 fits

Mejores hiperparámetros: {'subsample': 0.7, 'reg_lambda': 1.5, 'reg_alpha': 0, 'n_estimators': 400, 'min_child_weight': 3, 'max_depth': 12, 'learning_rate': 0.08, 'gamma': 0, 'colsample_bytree': 0.7}
Mejor ROC-AUC CV: 0.9567
Modelo XGBoost tuneado guardado en models/xgboost.pkl
